# PACClusterPermutationGLM Smoke + Batch Test

This notebook applies the same cluster-permutation GLM workflow used for TFRs to **trialwise 2D z-scored PAC maps** (phase-bin x amplitude-bin).

Model defaults:
- Event: `feedback_start`
- Predictor of interest: `zrpe`
- Covariate: `zpm`
- Cluster permutations: `200`

> Note: `200` permutations are for smoke/runtime practicality and are not final-inference settings.


In [ ]:
from pathlib import Path
import json
from datetime import datetime
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from IPython.display import display

from mv_regression_utils import (
    TFRClusterPermutationGLM,
    run_batch_crossregion_pac_cluster_glm_pipeline,
    process_subject_withinregion_pac_cluster_glm,
    run_batch_withinregion_pac_cluster_glm_pipeline,
    _default_filtered_subjects,
    _orient_trialwise_3d_map,
)
from plot_config import set_plot_style

warnings.filterwarnings('ignore')
set_plot_style()


In [ ]:
BASE_DIR = '/Volumes/T7 Shield'
LOAD_DIR = f'{BASE_DIR}/work/qasims01/MemoryBanditData/EMU'
PAC_DIR = f'{BASE_DIR}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac'

SUBJECT = 'MS036'
EVENT = 'feedback_start'
PREDICTOR = 'zrpe_cat'
COVARIATES = []

N_PERMUTATIONS = 200
ALTERNATIVE = 'two-sided'
ALPHA_CLUSTER = 0.05
ALPHA_TEST = 0.05
RANDOM_STATE = 42
MIN_ALIGNED_TRIALS = 20

OUTPUT_DIR = Path('/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-PACClusterPermutationGLM-smoketest')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PAC_TRIAL_CSV = Path(PAC_DIR) / 'all_subjects_crossregion_pac_combined.csv'

print('Configuration loaded.')
print(f'PAC trial table: {PAC_TRIAL_CSV}')
print(f'Output directory: {OUTPUT_DIR}')


In [ ]:
if not PAC_TRIAL_CSV.exists():
    raise FileNotFoundError(f'PAC combined trial CSV not found: {PAC_TRIAL_CSV}')

trial_df = pd.read_csv(PAC_TRIAL_CSV)
required_cols = [
    'participant', 'event', 'phase_roi', 'amp_roi',
    'phase_channel', 'amp_channel', 'hemi',
    'trials', PREDICTOR
] + COVARIATES
missing = [c for c in required_cols if c not in trial_df.columns]
if len(missing) > 0:
    raise ValueError(f'Missing required PAC columns: {missing}')

subj_df = trial_df[
    (trial_df['participant'].astype(str) == SUBJECT) &
    (trial_df['event'].astype(str) == EVENT)
].copy()

print('PAC trial diagnostics')
print('- total rows in combined table:', len(trial_df))
print(f'- rows for {SUBJECT} / {EVENT}:', len(subj_df))
print('- NaN counts in smoke regressors:')
print(subj_df[[PREDICTOR] + COVARIATES].isna().sum())


In [ ]:
def pac_file_for_pair(pac_dir, participant, phase_roi, amp_roi, phase_channel, amp_channel):
    return Path(pac_dir) / f'{participant}_{phase_roi}-{amp_roi}_{phase_channel}-{amp_channel}_pac.npy'


def align_pair_map_and_predictors(pair_df, pac_path, predictor, covariates):
    pair_df = pair_df.copy()
    pair_df['trials'] = pd.to_numeric(pair_df['trials'], errors='coerce')
    pair_df = pair_df.dropna(subset=['trials']).sort_values('trials')
    pair_df = pair_df.drop_duplicates(subset=['trials'], keep='last').reset_index(drop=True)

    pac_map = np.load(pac_path, mmap_mode='r')
    n_trial_hint = int(pair_df['trials'].max()) if len(pair_df) > 0 else None
    pac_trials_first = _orient_trialwise_3d_map(pac_map, n_trial_hint=n_trial_hint)
    if pac_trials_first is None or pac_trials_first.ndim != 3:
        return None, None, 'invalid_pac_shape'

    trial_ix = pair_df['trials'].to_numpy(dtype=int) - 1
    in_bounds = (trial_ix >= 0) & (trial_ix < int(pac_trials_first.shape[0]))

    reg_df = pair_df.loc[in_bounds, ['trials', predictor] + covariates].reset_index(drop=True)
    trial_ix_valid = trial_ix[in_bounds]

    predictor_df = reg_df[[predictor] + covariates].copy()
    finite_pred = np.isfinite(predictor_df.to_numpy(dtype=np.float64)).all(axis=1)
    predictor_df = predictor_df.loc[finite_pred].reset_index(drop=True)

    trial_ix_final = trial_ix_valid[finite_pred]
    map_aligned = np.asarray(pac_trials_first[trial_ix_final, :, :], dtype=np.float64)

    if len(predictor_df) == 0:
        return None, None, 'no_aligned_trials'
    if not np.isfinite(map_aligned).all():
        return None, None, 'pac_contains_nan'

    return map_aligned, predictor_df, None


pair_cols = ['participant', 'event', 'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel', 'hemi']
selected = None

for key_vals, gdf in subj_df.groupby(pair_cols, dropna=False):
    participant, ev, phase_roi, amp_roi, phase_channel, amp_channel, hemi = [str(v) for v in key_vals]
    pac_path = pac_file_for_pair(PAC_DIR, participant, phase_roi, amp_roi, phase_channel, amp_channel)
    if not pac_path.exists():
        continue

    map_aligned, predictor_df, reason = align_pair_map_and_predictors(
        gdf, pac_path, PREDICTOR, COVARIATES
    )
    if reason is not None:
        continue
    if len(predictor_df) < MIN_ALIGNED_TRIALS:
        continue

    selected = {
        'participant': participant,
        'event': ev,
        'phase_roi': phase_roi,
        'amp_roi': amp_roi,
        'phase_channel': phase_channel,
        'amp_channel': amp_channel,
        'hemi': hemi,
        'pac_path': pac_path,
        'map_aligned': map_aligned,
        'predictor_df': predictor_df,
    }
    break

if selected is None:
    raise RuntimeError(
        f'No valid PAC pair found for subject={SUBJECT}, event={EVENT} '
        f'with >= {MIN_ALIGNED_TRIALS} aligned finite trials.'
    )

print('Selected pair for smoke test')
print(f"- pair: {selected['phase_roi']}->{selected['amp_roi']} | {selected['phase_channel']}->{selected['amp_channel']} ({selected['hemi']})")
print(f"- PAC file: {selected['pac_path']}")
print(f"- aligned map shape: {selected['map_aligned'].shape}")
print(f"- predictor rows: {len(selected['predictor_df'])}")


In [ ]:
map_aligned = selected['map_aligned']
predictor_df = selected['predictor_df']

assert PREDICTOR in predictor_df.columns, f'Missing predictor column: {PREDICTOR}'
for cov in COVARIATES:
    assert cov in predictor_df.columns, f'Missing covariate: {cov}'
for col in predictor_df.columns:
    assert pd.api.types.is_numeric_dtype(predictor_df[col]), f'Non-numeric predictor/covariate: {col}'

assert map_aligned.ndim == 3, f'Expected 3D map (trials, dim0, dim1), got {map_aligned.shape}'
assert len(predictor_df) == map_aligned.shape[0], 'Predictor rows must equal map trials'
assert np.isfinite(predictor_df.to_numpy(dtype=np.float64)).all(), 'Predictor matrix has NaN/inf'
assert np.isfinite(map_aligned).all(), 'PAC map has NaN/inf'

print('Ready to run PAC GLM')
print(f"- subject: {SUBJECT}")
print(f"- event: {EVENT}")
print(f"- pair: {selected['phase_channel']} -> {selected['amp_channel']}")
print(f"- trials: {map_aligned.shape[0]} | phase bins: {map_aligned.shape[1]} | amp bins: {map_aligned.shape[2]}")


In [ ]:
glm = TFRClusterPermutationGLM(
    tfr_data=map_aligned,
    predictor_data=predictor_df,
    target_var=PREDICTOR,
    ch_name=f"{selected['phase_channel']}->{selected['amp_channel']}",
    alternative=ALTERNATIVE,
    alpha_cluster=ALPHA_CLUSTER,
    alpha_test=ALPHA_TEST,
    standardize_X=True,
    random_state=RANDOM_STATE,
)

res = glm.run(n_permutations=N_PERMUTATIONS, n_jobs=1, verbose=0)
print('PAC GLM smoke run complete.')


In [ ]:
clusters = res['clusters']
sig_clusters = res['significant_clusters']

print('Smoke summary')
print(f"- subject: {SUBJECT}")
print(f"- pair: {selected['phase_roi']}->{selected['amp_roi']} | {selected['phase_channel']}->{selected['amp_channel']} ({selected['hemi']})")
print(f"- df_resid: {res['df_resid']}")
print(f"- tcrit: {res['tcrit']:.4f}")
print(f"- suprathreshold clusters: {len(clusters)}")
print(f"- significant clusters (p < {ALPHA_TEST}): {len(sig_clusters)}")

top_rows = []
for c in clusters[:5]:
    top_rows.append({
        'sign': c.sign,
        'pvalue': float(c.pvalue),
        'abs_mass': float(c.abs_mass),
        'phase_idx': tuple(int(v) for v in c.freq_idx),
        'amp_idx': tuple(int(v) for v in c.time_idx),
        'n_pixels': int(c.n_pixels),
    })

cluster_table = pd.DataFrame(
    top_rows,
    columns=['sign', 'pvalue', 'abs_mass', 'phase_idx', 'amp_idx', 'n_pixels']
)
display(cluster_table)


In [ ]:
def symmetric_norm(arr):
    vmax = float(np.nanmax(np.abs(arr)))
    if not np.isfinite(vmax) or vmax <= 0:
        return None
    return TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)


def build_significant_mask(shape, significant_clusters):
    mask = np.zeros(shape, dtype=bool)
    for c in significant_clusters:
        p0, p1 = c.freq_idx
        a0, a1 = c.time_idx
        mask[p0:p1 + 1, a0:a1 + 1] = True
    return mask


beta_map = np.asarray(res['beta_map'], dtype=np.float64)
t_map = np.asarray(res['t_map'], dtype=np.float64)
sig_mask = build_significant_mask(t_map.shape, sig_clusters)

amp_bins = np.arange(t_map.shape[1], dtype=np.float64)
phase_bins = np.arange(t_map.shape[0], dtype=np.float64)
extent = [float(amp_bins.min()), float(amp_bins.max()), float(phase_bins.min()), float(phase_bins.max())]
dx = float(np.median(np.diff(amp_bins))) if len(amp_bins) > 1 else 1.0
dy = float(np.median(np.diff(phase_bins))) if len(phase_bins) > 1 else 1.0

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)

im0 = axes[0].imshow(
    beta_map,
    origin='lower',
    aspect='auto',
    extent=extent,
    cmap='RdBu_r',
    norm=symmetric_norm(beta_map),
)
axes[0].set_title('A. Beta Map (zrpe)')
axes[0].set_xlabel('Amplitude Bin')
axes[0].set_ylabel('Phase Bin')
cb0 = fig.colorbar(im0, ax=axes[0], shrink=0.9)
cb0.set_label('Beta')

im1 = axes[1].imshow(
    t_map,
    origin='lower',
    aspect='auto',
    extent=extent,
    cmap='RdBu_r',
    norm=symmetric_norm(t_map),
)
axes[1].set_title('B. t Map')
axes[1].set_xlabel('Amplitude Bin')
axes[1].set_ylabel('Phase Bin')
cb1 = fig.colorbar(im1, ax=axes[1], shrink=0.9)
cb1.set_label('t')

im2 = axes[2].imshow(
    t_map,
    origin='lower',
    aspect='auto',
    extent=extent,
    cmap='Greys',
    alpha=0.5,
)
overlay = np.ma.masked_where(~sig_mask, sig_mask.astype(float))
axes[2].imshow(
    overlay,
    origin='lower',
    aspect='auto',
    extent=extent,
    cmap='viridis',
    alpha=0.7,
)
for c in sig_clusters:
    p0, p1 = c.freq_idx
    a0, a1 = c.time_idx
    x0 = float(amp_bins[a0])
    y0 = float(phase_bins[p0])
    width = float(amp_bins[a1] - amp_bins[a0] + dx)
    height = float(phase_bins[p1] - phase_bins[p0] + dy)
    axes[2].add_patch(Rectangle((x0, y0), width, height, fill=False, edgecolor='yellow', linewidth=1.5))

if len(sig_clusters) == 0:
    axes[2].text(
        0.5,
        0.5,
        'No significant clusters',
        transform=axes[2].transAxes,
        ha='center',
        va='center',
        fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'),
    )

axes[2].set_title('C. Significant Cluster Mask')
axes[2].set_xlabel('Amplitude Bin')
axes[2].set_ylabel('Phase Bin')
cb2 = fig.colorbar(im2, ax=axes[2], shrink=0.9)
cb2.set_label('t (background)')

fig.suptitle(
    f"PAC GLM Smoke | {SUBJECT} | {selected['phase_channel']}->{selected['amp_channel']} | perms={N_PERMUTATIONS}",
    fontsize=11,
)

smoke_fig = OUTPUT_DIR / (
    f"pac_glm_smoke_{SUBJECT}_{selected['phase_channel']}_{selected['amp_channel']}_{EVENT}.png"
)
plt.savefig(smoke_fig, dpi=300, bbox_inches='tight')
plt.show()

print('Saved smoke figure:', smoke_fig)


In [ ]:
null_vals = np.asarray(res['null_max_abs_masses'], dtype=np.float64)
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.hist(null_vals, bins=30, color='steelblue', alpha=0.8, edgecolor='white')

if len(clusters) > 0:
    max_obs_mass = float(max(c.abs_mass for c in clusters))
    ax.axvline(
        max_obs_mass,
        color='crimson',
        linestyle='--',
        linewidth=2,
        label=f'max observed |cluster mass| = {max_obs_mass:.2f}',
    )
    ax.legend(loc='best', frameon=False)
else:
    ax.text(0.5, 0.9, 'No observed clusters', transform=ax.transAxes, ha='center', va='center')

ax.set_title('Null Distribution: Max |Cluster Mass|')
ax.set_xlabel('Max |cluster mass| per permutation')
ax.set_ylabel('Count')
sns.despine()
plt.tight_layout()
plt.show()


## Batch All-Subject / All-PAC-Pair Run (Checkpointed, Memory-Safe)

- Processes subjects sequentially, then PAC pairs sequentially within subject.
- Uses per-subject checkpoints for resumable execution.
- Saves plots only for pairs with `>=1` significant cluster by default.
- Designed to avoid loading many PAC maps in memory at once.


In [ ]:
BATCH_EVENT = 'feedback_start'
BATCH_PREDICTOR = 'zrpe'
BATCH_COVARIATES = ['zpm']
BATCH_N_PERMUTATIONS = 200
BATCH_ALPHA_CLUSTER = 0.05
BATCH_ALPHA_TEST = 0.05
BATCH_RANDOM_STATE = 42
BATCH_MIN_ALIGNED_TRIALS = 20

BATCH_RESUME = True
BATCH_OVERWRITE = False
BATCH_MAX_SUBJECTS = None   # set 1 or 2 for staged tests
BATCH_MAX_PAIRS_PER_SUBJECT = None
BATCH_SAVE_NON_SIG_PLOTS = False

BATCH_RUN_DIR = OUTPUT_DIR / f"batch_pac_glm_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print('Batch run directory:', BATCH_RUN_DIR)


In [ ]:
subjects = _default_filtered_subjects(BASE_DIR)

# Optional manual override examples:
# subjects = ['MS036']
# subjects = subjects[:2]

print(f'Filtered subjects: {len(subjects)}')
print('First 10:', subjects[:10])


In [ ]:
checkpoints_dir = BATCH_RUN_DIR / 'checkpoints'
existing_audit_rows = 0
if checkpoints_dir.exists():
    for pth in checkpoints_dir.glob('*_audit_pairs.csv'):
        try:
            existing_audit_rows += len(pd.read_csv(pth))
        except Exception:
            pass

print('Preflight')
print('- subjects requested:', len(subjects))
print('- output run dir:', BATCH_RUN_DIR)
print('- PAC trial table:', PAC_TRIAL_CSV)
print('- resume:', BATCH_RESUME, '| overwrite:', BATCH_OVERWRITE)
print('- max_subjects:', BATCH_MAX_SUBJECTS)
print('- max_pairs_per_subject:', BATCH_MAX_PAIRS_PER_SUBJECT)
print('- save_non_sig_plots:', BATCH_SAVE_NON_SIG_PLOTS)
print('- existing audit rows in run dir:', existing_audit_rows)


In [ ]:
batch_out = run_batch_crossregion_pac_cluster_glm_pipeline(
    subjects=subjects,
    base_dir=BASE_DIR,
    load_dir=LOAD_DIR,
    pac_dir=PAC_DIR,
    run_dir=str(BATCH_RUN_DIR),
    trial_df=str(PAC_TRIAL_CSV),
    event=BATCH_EVENT,
    predictor=BATCH_PREDICTOR,
    covariates=BATCH_COVARIATES,
    n_permutations=BATCH_N_PERMUTATIONS,
    alpha_cluster=BATCH_ALPHA_CLUSTER,
    alpha_test=BATCH_ALPHA_TEST,
    alternative='two-sided',
    min_aligned_trials=BATCH_MIN_ALIGNED_TRIALS,
    random_state=BATCH_RANDOM_STATE,
    overwrite=BATCH_OVERWRITE,
    resume=BATCH_RESUME,
    max_subjects=BATCH_MAX_SUBJECTS,
    max_pairs_per_subject=BATCH_MAX_PAIRS_PER_SUBJECT,
    save_non_sig_plots=BATCH_SAVE_NON_SIG_PLOTS,
)

sig_pairs_df = batch_out['sig_pairs']
sig_electrodes_df = batch_out['sig_electrodes']
region_df = batch_out['region_counts']
audit_pairs_df = batch_out['audit_pairs']

print('Batch run complete.')
print('- significant pair rows:', len(sig_pairs_df))
print('- significant electrode rows:', len(sig_electrodes_df))
print('- region rows:', len(region_df))
print('- audit pair rows:', len(audit_pairs_df))


In [ ]:
# Build tested-vs-significant proportion summaries for ROI plots
pair_id_cols = ['subject', 'event', 'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel', 'hemi']
missing_audit_cols = [c for c in (['status'] + pair_id_cols) if c not in audit_pairs_df.columns]
if len(missing_audit_cols) > 0:
    raise ValueError(f'Missing required audit columns for proportion summaries: {missing_audit_cols}')

missing_sig_cols = [c for c in pair_id_cols if c not in sig_pairs_df.columns]
if len(sig_pairs_df) > 0 and len(missing_sig_cols) > 0:
    raise ValueError(f'Missing required significant-pair columns for proportion summaries: {missing_sig_cols}')

tested_pairs_df = (
    audit_pairs_df[audit_pairs_df['status'].astype(str) == 'ok'][pair_id_cols]
    .copy()
    .drop_duplicates()
)
tested_pairs_df['roi_pair'] = tested_pairs_df['phase_roi'].astype(str) + '->' + tested_pairs_df['amp_roi'].astype(str)

if len(sig_pairs_df) > 0:
    sig_pairs_unique_df = sig_pairs_df[pair_id_cols].copy().drop_duplicates()
else:
    sig_pairs_unique_df = pd.DataFrame(columns=pair_id_cols)
sig_pairs_unique_df['roi_pair'] = (
    sig_pairs_unique_df['phase_roi'].astype(str) + '->' + sig_pairs_unique_df['amp_roi'].astype(str)
    if len(sig_pairs_unique_df) > 0
    else pd.Series(dtype=str)
)

if len(tested_pairs_df) == 0:
    cross_roi_pair_prop_df = pd.DataFrame(columns=['roi_pair', 'n_sig_pairs', 'n_tested_pairs', 'prop_sig_pairs'])
    cross_roi_prop_df = pd.DataFrame(columns=['roi', 'n_sig_pairs_by_roi', 'n_tested_pairs_by_roi', 'prop_sig_pairs_by_roi'])
    print('ROI Proportion Summary (tested pairs only; status=ok)')
    print('No tested pairs (status==ok) available. Skipping proportion summaries.')
else:
    tested_pair_counts = tested_pairs_df.groupby('roi_pair').size().reset_index(name='n_tested_pairs')
    sig_pair_counts = sig_pairs_unique_df.groupby('roi_pair').size().reset_index(name='n_sig_pairs') if len(sig_pairs_unique_df) > 0 else pd.DataFrame(columns=['roi_pair', 'n_sig_pairs'])

    cross_roi_pair_prop_df = tested_pair_counts.merge(sig_pair_counts, on='roi_pair', how='left')
    cross_roi_pair_prop_df['n_sig_pairs'] = cross_roi_pair_prop_df['n_sig_pairs'].fillna(0).astype(int)
    cross_roi_pair_prop_df['prop_sig_pairs'] = cross_roi_pair_prop_df['n_sig_pairs'] / cross_roi_pair_prop_df['n_tested_pairs']
    cross_roi_pair_prop_df = cross_roi_pair_prop_df[cross_roi_pair_prop_df['n_tested_pairs'] > 0].copy()
    cross_roi_pair_prop_df = cross_roi_pair_prop_df.sort_values('prop_sig_pairs', ascending=False).reset_index(drop=True)

    tested_roi_long = pd.concat([
        tested_pairs_df[['subject', 'event', 'phase_roi', 'phase_channel', 'hemi']].rename(columns={'phase_roi': 'roi', 'phase_channel': 'electrode'}),
        tested_pairs_df[['subject', 'event', 'amp_roi', 'amp_channel', 'hemi']].rename(columns={'amp_roi': 'roi', 'amp_channel': 'electrode'}),
    ], ignore_index=True)

    sig_roi_long = pd.concat([
        sig_pairs_unique_df[['subject', 'event', 'phase_roi', 'phase_channel', 'hemi']].rename(columns={'phase_roi': 'roi', 'phase_channel': 'electrode'}),
        sig_pairs_unique_df[['subject', 'event', 'amp_roi', 'amp_channel', 'hemi']].rename(columns={'amp_roi': 'roi', 'amp_channel': 'electrode'}),
    ], ignore_index=True) if len(sig_pairs_unique_df) > 0 else pd.DataFrame(columns=['subject', 'event', 'roi', 'electrode', 'hemi'])

    tested_roi_counts = tested_roi_long.groupby('roi').size().reset_index(name='n_tested_pairs_by_roi')
    sig_roi_counts = sig_roi_long.groupby('roi').size().reset_index(name='n_sig_pairs_by_roi') if len(sig_roi_long) > 0 else pd.DataFrame(columns=['roi', 'n_sig_pairs_by_roi'])

    cross_roi_prop_df = tested_roi_counts.merge(sig_roi_counts, on='roi', how='left')
    cross_roi_prop_df['n_sig_pairs_by_roi'] = cross_roi_prop_df['n_sig_pairs_by_roi'].fillna(0).astype(int)
    cross_roi_prop_df['prop_sig_pairs_by_roi'] = cross_roi_prop_df['n_sig_pairs_by_roi'] / cross_roi_prop_df['n_tested_pairs_by_roi']
    cross_roi_prop_df = cross_roi_prop_df[cross_roi_prop_df['n_tested_pairs_by_roi'] > 0].copy()
    cross_roi_prop_df = cross_roi_prop_df.sort_values('prop_sig_pairs_by_roi', ascending=False).reset_index(drop=True)

    print('ROI Proportion Summary (tested pairs only; status=ok)')
    display(cross_roi_prop_df)
    print('\nROI-Pair Proportion Summary (tested pairs only; status=ok)')
    display(cross_roi_pair_prop_df)


In [ ]:
sig_pairs_csv = BATCH_RUN_DIR / 'sig_pac_pairs.csv'
sig_electrodes_csv = BATCH_RUN_DIR / 'sig_electrodes.csv'
region_csv = BATCH_RUN_DIR / 'region_counts.csv'
audit_csv = BATCH_RUN_DIR / 'all_pac_pair_audit.csv'
meta_json = BATCH_RUN_DIR / 'run_metadata.json'

print('Artifacts')
for p in [sig_pairs_csv, sig_electrodes_csv, region_csv, audit_csv, meta_json]:
    print(f'- {p.name}: exists={p.exists()}')

if sig_pairs_csv.exists():
    print('sig_pac_pairs.csv rows:', len(pd.read_csv(sig_pairs_csv)))
if sig_electrodes_csv.exists():
    print('sig_electrodes.csv rows:', len(pd.read_csv(sig_electrodes_csv)))
if region_csv.exists():
    print('region_counts.csv rows:', len(pd.read_csv(region_csv)))
if audit_csv.exists():
    print('all_pac_pair_audit.csv rows:', len(pd.read_csv(audit_csv)))

if meta_json.exists():
    with open(meta_json, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=2))


In [ ]:
sig_pair_cols = [
    'subject', 'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel',
    'n_sig_clusters', 'min_sig_p', 'plot_path'
]
region_cols = ['region', 'n_sig_electrodes', 'n_subjects_with_sig']

print('Significant PAC pairs (core output)')
if len(sig_pairs_df) > 0:
    display(sig_pairs_df.loc[:, [c for c in sig_pair_cols if c in sig_pairs_df.columns]])
else:
    print('No significant PAC pairs found.')

print('\nRaw region counts artifact (electrode-level, not proportion)')
if len(region_df) > 0:
    display(region_df.loc[:, [c for c in region_cols if c in region_df.columns]])
else:
    print('No region counts available.')


In [ ]:
if len(cross_roi_prop_df) > 0:
    roi_plot_df = cross_roi_prop_df.sort_values('prop_sig_pairs_by_roi', ascending=False).copy()
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    sns.barplot(data=roi_plot_df, x='roi', y='prop_sig_pairs_by_roi', color='steelblue', ax=ax)
    ax.set_title('Cross-Region PAC: Proportion of Significant Pairs by ROI')
    ax.set_xlabel('ROI')
    ax.set_ylabel('Proportion Significant (n_sig / n_tested)')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)
    for patch, (_, row) in zip(ax.patches, roi_plot_df.iterrows()):
        x = patch.get_x() + patch.get_width() / 2.0
        y = patch.get_height()
        ax.text(x, min(1.0, y + 0.02), f"{int(row['n_sig_pairs_by_roi'])}/{int(row['n_tested_pairs_by_roi'])}", ha='center', va='bottom', fontsize=8)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print('Skipping ROI proportion plot: no tested pairs with status==ok.')

if len(cross_roi_pair_prop_df) > 0:
    roi_pair_plot_df = cross_roi_pair_prop_df.sort_values('prop_sig_pairs', ascending=False).copy()
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    sns.barplot(data=roi_pair_plot_df, x='roi_pair', y='prop_sig_pairs', color='darkgreen', ax=ax)
    ax.set_title('Cross-Region PAC: Proportion of Significant Pairs by ROI Pair')
    ax.set_xlabel('Phase ROI -> Amp ROI')
    ax.set_ylabel('Proportion Significant (n_sig / n_tested)')
    ax.set_ylim(0, 0.2)
    ax.tick_params(axis='x', rotation=45)
    for patch, (_, row) in zip(ax.patches, roi_pair_plot_df.iterrows()):
        x = patch.get_x() + patch.get_width() / 2.0
        y = patch.get_height()
        ax.text(x, min(1.0, y + 0.02), f"{int(row['n_sig_pairs'])}/{int(row['n_tested_pairs'])}", ha='center', va='bottom', fontsize=8)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print('Skipping ROI-pair proportion plot: no tested pairs with status==ok.')

if len(sig_electrodes_df) > 0 and 'subject' in sig_electrodes_df.columns:
    subj_counts = (
        sig_electrodes_df.groupby('subject')['electrode']
        .nunique()
        .sort_values(ascending=False)
        .reset_index(name='n_sig_electrodes')
    )
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    sns.barplot(data=subj_counts, x='subject', y='n_sig_electrodes', color='darkorange', ax=ax)
    ax.set_title('Significant Electrodes by Subject (PAC)')
    ax.set_xlabel('Subject')
    ax.set_ylabel('# Significant Electrodes')
    ax.tick_params(axis='x', rotation=45)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print('Skipping subject bar plot: sig_electrodes_df is empty.')


### Resume Workflow Notes

- Continue interrupted run: keep the same `BATCH_RUN_DIR`, set `BATCH_RESUME=True`, `BATCH_OVERWRITE=False`.
- Force full rerun in same folder: set `BATCH_OVERWRITE=True` (or choose a new `BATCH_RUN_DIR`).
- Stage safely: start with `BATCH_MAX_SUBJECTS=1`, then `2`, then `None`.
- If you want true cross-session resume, set `BATCH_RUN_DIR` to a fixed non-timestamp path.


## Within-Region PAC Cluster GLM (Smoke + Batch, Checkpointed)

- Uses within-electrode PAC exports from `MemoryBandit-WithinRegionPAC.ipynb`.
- Runs one smoke electrode, then a memory-safe checkpointed batch across subjects/electrodes.
- Saves significant-only plots by default.


In [ ]:
WITHIN_PAC_TRIAL_CSV = Path(PAC_DIR) / 'all_subjects_withinelectrode_pac_combined.csv'
WITHIN_EVENT = 'feedback_start'
WITHIN_PREDICTOR = 'zrpe'
WITHIN_COVARIATES = ['zpm']
WITHIN_N_PERMUTATIONS = 200
WITHIN_ALPHA_CLUSTER = 0.05
WITHIN_ALPHA_TEST = 0.05
WITHIN_RANDOM_STATE = 42
WITHIN_MIN_ALIGNED_TRIALS = 20

WITHIN_SUBJECT = 'MS036'
print('Within config loaded.')
print(f'Within trial CSV: {WITHIN_PAC_TRIAL_CSV}')


In [ ]:
if not WITHIN_PAC_TRIAL_CSV.exists():
    raise FileNotFoundError(f'Within PAC combined trial CSV not found: {WITHIN_PAC_TRIAL_CSV}')

within_trial_df = pd.read_csv(WITHIN_PAC_TRIAL_CSV)
within_req = [
    'participant', 'event', 'phase_roi', 'amp_roi',
    'phase_channel', 'amp_channel', 'hemi', 'trials',
    WITHIN_PREDICTOR
] + WITHIN_COVARIATES
within_missing = [c for c in within_req if c not in within_trial_df.columns]
if len(within_missing) > 0:
    raise ValueError(f'Missing required within-PAC columns: {within_missing}')

within_subj_df = within_trial_df[
    (within_trial_df['participant'].astype(str) == WITHIN_SUBJECT) &
    (within_trial_df['event'].astype(str) == WITHIN_EVENT)
].copy()

print('Within trial diagnostics')
print('- total rows in combined table:', len(within_trial_df))
print(f'- rows for {WITHIN_SUBJECT} / {WITHIN_EVENT}:', len(within_subj_df))


In [ ]:
def within_pac_file_for_unit(pac_dir, participant, region, electrode):
    return Path(pac_dir) / f'{participant}_{region}_WITHIN_{electrode}_pac.npy'


def align_within_map_and_predictors(unit_df, pac_path, predictor, covariates):
    unit_df = unit_df.copy()
    unit_df['trials'] = pd.to_numeric(unit_df['trials'], errors='coerce')
    unit_df = unit_df.dropna(subset=['trials']).sort_values('trials')
    unit_df = unit_df.drop_duplicates(subset=['trials'], keep='last').reset_index(drop=True)

    pac_map = np.load(pac_path, mmap_mode='r')
    n_trial_hint = int(unit_df['trials'].max()) if len(unit_df) > 0 else None
    pac_trials_first = _orient_trialwise_3d_map(pac_map, n_trial_hint=n_trial_hint)
    if pac_trials_first is None or pac_trials_first.ndim != 3:
        return None, None, 'invalid_pac_shape'

    trial_ix = unit_df['trials'].to_numpy(dtype=int) - 1
    in_bounds = (trial_ix >= 0) & (trial_ix < int(pac_trials_first.shape[0]))

    reg_df = unit_df.loc[in_bounds, ['trials', predictor] + covariates].reset_index(drop=True)
    trial_ix_valid = trial_ix[in_bounds]

    predictor_df = reg_df[[predictor] + covariates].copy()
    finite_pred = np.isfinite(predictor_df.to_numpy(dtype=np.float64)).all(axis=1)
    predictor_df = predictor_df.loc[finite_pred].reset_index(drop=True)

    trial_ix_final = trial_ix_valid[finite_pred]
    map_aligned = np.asarray(pac_trials_first[trial_ix_final, :, :], dtype=np.float64)

    if len(predictor_df) == 0:
        return None, None, 'no_aligned_trials'
    if not np.isfinite(map_aligned).all():
        return None, None, 'pac_contains_nan'

    return map_aligned, predictor_df, None


within_unit_cols = ['participant', 'event', 'phase_roi', 'amp_roi', 'phase_channel', 'amp_channel', 'hemi']
within_selected = None

for key_vals, gdf in within_subj_df.groupby(within_unit_cols, dropna=False):
    participant, ev, phase_roi, amp_roi, phase_channel, amp_channel, hemi = [str(v) for v in key_vals]
    if (phase_roi != amp_roi) or (phase_channel != amp_channel):
        continue

    pac_path = within_pac_file_for_unit(PAC_DIR, participant, phase_roi, phase_channel)
    if not pac_path.exists():
        continue

    map_aligned, predictor_df, reason = align_within_map_and_predictors(
        gdf, pac_path, WITHIN_PREDICTOR, WITHIN_COVARIATES
    )
    if reason is not None:
        continue
    if len(predictor_df) < WITHIN_MIN_ALIGNED_TRIALS:
        continue

    within_selected = {
        'participant': participant,
        'event': ev,
        'region': phase_roi,
        'electrode': phase_channel,
        'hemi': hemi,
        'pac_path': pac_path,
        'map_aligned': map_aligned,
        'predictor_df': predictor_df,
    }
    break

if within_selected is None:
    raise RuntimeError(
        f'No valid within PAC electrode found for subject={WITHIN_SUBJECT}, event={WITHIN_EVENT} '
        f'with >= {WITHIN_MIN_ALIGNED_TRIALS} aligned finite trials.'
    )

print('Selected within electrode for smoke test')
print(f"- region/electrode: {within_selected['region']} / {within_selected['electrode']} ({within_selected['hemi']})")
print(f"- PAC file: {within_selected['pac_path']}")
print(f"- aligned map shape: {within_selected['map_aligned'].shape}")
print(f"- predictor rows: {len(within_selected['predictor_df'])}")


In [ ]:
within_map_aligned = within_selected['map_aligned']
within_predictor_df = within_selected['predictor_df']

assert WITHIN_PREDICTOR in within_predictor_df.columns, f'Missing predictor column: {WITHIN_PREDICTOR}'
for cov in WITHIN_COVARIATES:
    assert cov in within_predictor_df.columns, f'Missing covariate: {cov}'
assert len(within_predictor_df) == within_map_aligned.shape[0], 'Predictor rows must equal map trials'
assert np.isfinite(within_predictor_df.to_numpy(dtype=np.float64)).all(), 'Predictor matrix has NaN/inf'
assert np.isfinite(within_map_aligned).all(), 'Within PAC map has NaN/inf'

within_glm = TFRClusterPermutationGLM(
    tfr_data=within_map_aligned,
    predictor_data=within_predictor_df,
    target_var=WITHIN_PREDICTOR,
    ch_name=f"{within_selected['region']}:{within_selected['electrode']}",
    alternative='two-sided',
    alpha_cluster=WITHIN_ALPHA_CLUSTER,
    alpha_test=WITHIN_ALPHA_TEST,
    standardize_X=True,
    random_state=WITHIN_RANDOM_STATE,
)

within_res = within_glm.run(n_permutations=WITHIN_N_PERMUTATIONS, n_jobs=1, verbose=0)
print('Within PAC GLM smoke run complete.')


In [ ]:
within_clusters = within_res['clusters']
within_sig_clusters = within_res['significant_clusters']

print('Within smoke summary')
print(f"- subject: {WITHIN_SUBJECT}")
print(f"- electrode: {within_selected['region']} / {within_selected['electrode']} ({within_selected['hemi']})")
print(f"- df_resid: {within_res['df_resid']}")
print(f"- tcrit: {within_res['tcrit']:.4f}")
print(f"- suprathreshold clusters: {len(within_clusters)}")
print(f"- significant clusters (p < {WITHIN_ALPHA_TEST}): {len(within_sig_clusters)}")

within_top_rows = []
for c in within_clusters[:5]:
    within_top_rows.append({
        'sign': c.sign,
        'pvalue': float(c.pvalue),
        'abs_mass': float(c.abs_mass),
        'phase_idx': tuple(int(v) for v in c.freq_idx),
        'amp_idx': tuple(int(v) for v in c.time_idx),
        'n_pixels': int(c.n_pixels),
    })

within_cluster_table = pd.DataFrame(
    within_top_rows,
    columns=['sign', 'pvalue', 'abs_mass', 'phase_idx', 'amp_idx', 'n_pixels']
)
display(within_cluster_table)


In [ ]:
within_beta_map = np.asarray(within_res['beta_map'], dtype=np.float64)
within_t_map = np.asarray(within_res['t_map'], dtype=np.float64)
within_sig_mask = build_significant_mask(within_t_map.shape, within_sig_clusters)

within_amp_bins = np.arange(within_t_map.shape[1], dtype=np.float64)
within_phase_bins = np.arange(within_t_map.shape[0], dtype=np.float64)
within_extent = [
    float(within_amp_bins.min()),
    float(within_amp_bins.max()),
    float(within_phase_bins.min()),
    float(within_phase_bins.max()),
]
within_dx = float(np.median(np.diff(within_amp_bins))) if len(within_amp_bins) > 1 else 1.0
within_dy = float(np.median(np.diff(within_phase_bins))) if len(within_phase_bins) > 1 else 1.0

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)

im0 = axes[0].imshow(within_beta_map, origin='lower', aspect='auto', extent=within_extent, cmap='RdBu_r', norm=symmetric_norm(within_beta_map))
axes[0].set_title('A. Beta Map (zrpe)')
axes[0].set_xlabel('Amplitude Bin')
axes[0].set_ylabel('Phase Bin')
fig.colorbar(im0, ax=axes[0], shrink=0.9)

im1 = axes[1].imshow(within_t_map, origin='lower', aspect='auto', extent=within_extent, cmap='RdBu_r', norm=symmetric_norm(within_t_map))
axes[1].set_title('B. t Map')
axes[1].set_xlabel('Amplitude Bin')
axes[1].set_ylabel('Phase Bin')
fig.colorbar(im1, ax=axes[1], shrink=0.9)

im2 = axes[2].imshow(within_t_map, origin='lower', aspect='auto', extent=within_extent, cmap='Greys', alpha=0.5)
overlay = np.ma.masked_where(~within_sig_mask, within_sig_mask.astype(float))
axes[2].imshow(overlay, origin='lower', aspect='auto', extent=within_extent, cmap='viridis', alpha=0.7)
for c in within_sig_clusters:
    p0, p1 = c.freq_idx
    a0, a1 = c.time_idx
    x0 = float(within_amp_bins[a0])
    y0 = float(within_phase_bins[p0])
    width = float(within_amp_bins[a1] - within_amp_bins[a0] + within_dx)
    height = float(within_phase_bins[p1] - within_phase_bins[p0] + within_dy)
    axes[2].add_patch(Rectangle((x0, y0), width, height, fill=False, edgecolor='yellow', linewidth=1.5))

if len(within_sig_clusters) == 0:
    axes[2].text(0.5, 0.5, 'No significant clusters', transform=axes[2].transAxes, ha='center', va='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray'))

axes[2].set_title('C. Significant Cluster Mask')
axes[2].set_xlabel('Amplitude Bin')
axes[2].set_ylabel('Phase Bin')
fig.colorbar(im2, ax=axes[2], shrink=0.9)

fig.suptitle(
    f"Within PAC GLM Smoke | {WITHIN_SUBJECT} | {within_selected['region']}:{within_selected['electrode']} | perms={WITHIN_N_PERMUTATIONS}",
    fontsize=11,
)

within_smoke_fig = OUTPUT_DIR / f"within_pac_glm_smoke_{WITHIN_SUBJECT}_{within_selected['region']}_{within_selected['electrode']}_{WITHIN_EVENT}.png"
plt.savefig(within_smoke_fig, dpi=300, bbox_inches='tight')
plt.show()

print('Saved within smoke figure:', within_smoke_fig)

within_null_vals = np.asarray(within_res['null_max_abs_masses'], dtype=np.float64)
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.hist(within_null_vals, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
if len(within_clusters) > 0:
    within_max_obs_mass = float(max(c.abs_mass for c in within_clusters))
    ax.axvline(within_max_obs_mass, color='crimson', linestyle='--', linewidth=2, label=f'max observed |cluster mass| = {within_max_obs_mass:.2f}')
    ax.legend(loc='best', frameon=False)
ax.set_title('Within PAC Null Distribution: Max |Cluster Mass|')
ax.set_xlabel('Max |cluster mass| per permutation')
ax.set_ylabel('Count')
sns.despine()
plt.tight_layout()
plt.show()


### Within-Region Batch Run

- Subject-sequential and electrode-sequential execution.
- Resumable checkpoints and significant-only plots by default.
- Use throttles for staged runs before full cohort.


In [ ]:
WITHIN_BATCH_RESUME = True
WITHIN_BATCH_OVERWRITE = False
WITHIN_BATCH_MAX_SUBJECTS = None  # set 1 or 2 for staged tests
WITHIN_BATCH_MAX_ELECTRODES_PER_SUBJECT = None
WITHIN_BATCH_SAVE_NON_SIG_PLOTS = False

WITHIN_BATCH_RUN_DIR = OUTPUT_DIR / f"batch_within_pac_glm_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print('Within batch run dir:', WITHIN_BATCH_RUN_DIR)


In [ ]:
within_subjects = _default_filtered_subjects(BASE_DIR)

print(f'Filtered subjects: {len(within_subjects)}')
print('First 10:', within_subjects[:10])
print('Preflight')
print('- run dir:', WITHIN_BATCH_RUN_DIR)
print('- trial csv:', WITHIN_PAC_TRIAL_CSV)
print('- resume:', WITHIN_BATCH_RESUME, '| overwrite:', WITHIN_BATCH_OVERWRITE)
print('- max_subjects:', WITHIN_BATCH_MAX_SUBJECTS)
print('- max_electrodes_per_subject:', WITHIN_BATCH_MAX_ELECTRODES_PER_SUBJECT)
print('- save_non_sig_plots:', WITHIN_BATCH_SAVE_NON_SIG_PLOTS)


In [ ]:
within_batch_out = run_batch_withinregion_pac_cluster_glm_pipeline(
    subjects=within_subjects,
    base_dir=BASE_DIR,
    load_dir=LOAD_DIR,
    pac_dir=PAC_DIR,
    run_dir=str(WITHIN_BATCH_RUN_DIR),
    trial_df=str(WITHIN_PAC_TRIAL_CSV),
    event=WITHIN_EVENT,
    predictor=WITHIN_PREDICTOR,
    covariates=WITHIN_COVARIATES,
    n_permutations=WITHIN_N_PERMUTATIONS,
    alpha_cluster=WITHIN_ALPHA_CLUSTER,
    alpha_test=WITHIN_ALPHA_TEST,
    alternative='two-sided',
    min_aligned_trials=WITHIN_MIN_ALIGNED_TRIALS,
    random_state=WITHIN_RANDOM_STATE,
    overwrite=WITHIN_BATCH_OVERWRITE,
    resume=WITHIN_BATCH_RESUME,
    max_subjects=WITHIN_BATCH_MAX_SUBJECTS,
    max_electrodes_per_subject=WITHIN_BATCH_MAX_ELECTRODES_PER_SUBJECT,
    save_non_sig_plots=WITHIN_BATCH_SAVE_NON_SIG_PLOTS,
)

within_sig_df = within_batch_out['sig_electrodes']
within_region_df = within_batch_out['region_counts']
within_audit_df = within_batch_out['audit']

print('Within batch run complete.')
print('- significant electrodes:', len(within_sig_df))
print('- region rows:', len(within_region_df))
print('- audit rows:', len(within_audit_df))


In [ ]:
# Build within-region tested-vs-significant proportion summary
within_id_cols = ['subject', 'event', 'region', 'electrode', 'hemi']
missing_within_audit = [c for c in (['status'] + within_id_cols) if c not in within_audit_df.columns]
if len(missing_within_audit) > 0:
    raise ValueError(f'Missing required within audit columns for proportion summaries: {missing_within_audit}')

missing_within_sig = [c for c in within_id_cols if c not in within_sig_df.columns]
if len(within_sig_df) > 0 and len(missing_within_sig) > 0:
    raise ValueError(f'Missing required within significant columns for proportion summaries: {missing_within_sig}')

within_tested_df = (
    within_audit_df[within_audit_df['status'].astype(str) == 'ok'][within_id_cols]
    .copy()
    .drop_duplicates()
)
within_sig_unique_df = within_sig_df[within_id_cols].copy().drop_duplicates() if len(within_sig_df) > 0 else pd.DataFrame(columns=within_id_cols)

if len(within_tested_df) == 0:
    within_roi_prop_df = pd.DataFrame(columns=['region', 'n_sig_units', 'n_tested_units', 'prop_sig_units'])
    print('Within ROI Proportion Summary (tested electrodes only; status=ok)')
    print('No tested within electrodes (status==ok) available. Skipping proportion summary.')
else:
    within_tested_counts = within_tested_df.groupby('region').size().reset_index(name='n_tested_units')
    within_sig_counts = within_sig_unique_df.groupby('region').size().reset_index(name='n_sig_units') if len(within_sig_unique_df) > 0 else pd.DataFrame(columns=['region', 'n_sig_units'])

    within_roi_prop_df = within_tested_counts.merge(within_sig_counts, on='region', how='left')
    within_roi_prop_df['n_sig_units'] = within_roi_prop_df['n_sig_units'].fillna(0).astype(int)
    within_roi_prop_df['prop_sig_units'] = within_roi_prop_df['n_sig_units'] / within_roi_prop_df['n_tested_units']
    within_roi_prop_df = within_roi_prop_df[within_roi_prop_df['n_tested_units'] > 0].copy()
    within_roi_prop_df = within_roi_prop_df.sort_values('prop_sig_units', ascending=False).reset_index(drop=True)

    print('Within ROI Proportion Summary (tested electrodes only; status=ok)')
    display(within_roi_prop_df)


In [ ]:
within_sig_csv = WITHIN_BATCH_RUN_DIR / 'sig_within_pac_electrodes.csv'
within_region_csv = WITHIN_BATCH_RUN_DIR / 'region_counts_within.csv'
within_audit_csv = WITHIN_BATCH_RUN_DIR / 'all_within_pac_audit.csv'
within_meta_json = WITHIN_BATCH_RUN_DIR / 'run_metadata_within.json'

print('Within artifacts')
for pth in [within_sig_csv, within_region_csv, within_audit_csv, within_meta_json]:
    print(f'- {pth.name}: exists={pth.exists()}')

if within_sig_csv.exists():
    print('sig_within_pac_electrodes.csv rows:', len(pd.read_csv(within_sig_csv)))
if within_region_csv.exists():
    print('region_counts_within.csv rows:', len(pd.read_csv(within_region_csv)))
if within_audit_csv.exists():
    print('all_within_pac_audit.csv rows:', len(pd.read_csv(within_audit_csv)))

if within_meta_json.exists():
    with open(within_meta_json, 'r', encoding='utf-8') as f:
        within_meta = json.load(f)
    print(json.dumps(within_meta, indent=2))


In [ ]:
within_sig_view_cols = ['subject', 'electrode', 'region', 'n_sig_clusters', 'min_sig_p', 'plot_path']
within_region_view_cols = ['region', 'n_sig_electrodes', 'n_subjects_with_sig']

print('Significant within electrodes (core output)')
if len(within_sig_df) > 0:
    display(within_sig_df.loc[:, [c for c in within_sig_view_cols if c in within_sig_df.columns]])
else:
    print('No significant within electrodes found.')

print('\nRaw within region counts artifact (electrode-level, not proportion)')
if len(within_region_df) > 0:
    display(within_region_df.loc[:, [c for c in within_region_view_cols if c in within_region_df.columns]])
else:
    print('No within region counts available.')


In [ ]:
if len(within_roi_prop_df) > 0:
    within_region_plot_df = within_roi_prop_df.sort_values('prop_sig_units', ascending=False).copy()
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    sns.barplot(data=within_region_plot_df, x='region', y='prop_sig_units', color='steelblue', ax=ax)
    ax.set_title('Within PAC: Proportion of Significant Pairs by ROI')
    ax.set_xlabel('Region')
    ax.set_ylabel('Proportion Significant (n_sig / n_tested)')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)
    for patch, (_, row) in zip(ax.patches, within_region_plot_df.iterrows()):
        x = patch.get_x() + patch.get_width() / 2.0
        y = patch.get_height()
        ax.text(x, min(1.0, y + 0.02), f"{int(row['n_sig_units'])}/{int(row['n_tested_units'])}", ha='center', va='bottom', fontsize=8)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print('Skipping within ROI proportion plot: no tested within electrodes with status==ok.')

if len(within_sig_df) > 0 and 'subject' in within_sig_df.columns:
    within_subj_counts = (
        within_sig_df.groupby('subject')['electrode']
        .nunique()
        .sort_values(ascending=False)
        .reset_index(name='n_sig_electrodes')
    )
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    sns.barplot(data=within_subj_counts, x='subject', y='n_sig_electrodes', color='darkorange', ax=ax)
    ax.set_title('Within PAC: Significant Electrodes by Subject')
    ax.set_xlabel('Subject')
    ax.set_ylabel('# Significant Electrodes')
    ax.tick_params(axis='x', rotation=45)
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print('Skipping within subject bar plot: within_sig_df is empty.')


### Within Resume Workflow Notes

- Continue interrupted run: keep the same `WITHIN_BATCH_RUN_DIR`, set `WITHIN_BATCH_RESUME=True`, `WITHIN_BATCH_OVERWRITE=False`.
- Force full rerun in same folder: set `WITHIN_BATCH_OVERWRITE=True` (or choose a new run directory).
- Stage safely: start with `WITHIN_BATCH_MAX_SUBJECTS=1`, then `2`, then `None`.
